# 01 - Dataset Analysis & Integrity Verification

**Project:** Early-Stage Ransomware Detection via Dynamic Memory Analysis and Reinforcement Learning  
**Notebook 01:** Comprehensive Dataset Validation, Class Imbalance, and Data Leakage Investigation  

This notebook establishes empirical rigor by validating data integrity invariants, auditing class distributions, and investigating feature semantics and potential target leakage.


## 1. Setup & Environment
Ensure project root is in Python path and import modular routines from `src`.


In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import validate_dataset, EXPECTED_CLASSES
from src.leakage_investigation import run_leakage_analysis
from src.utils import get_data_dir, get_reports_dir

print(f"Project root: {project_root}")


## 2. Dataset Loading and Invariant Validation
Validate dataset shape ($1750 \times 47$), target column (`family`), missing values (0), infinite values (0), duplicates (0), and constant features (0).


In [ ]:
# Run automated validation suite
validation_report = validate_dataset()


## 3. Class Distribution & Imbalance
Analyze the representation of the three ransomware families: `G`, `E`, and `L`.


In [ ]:
data_path = get_data_dir() / "processed" / "cleaned_dataset.csv"
df = pd.read_csv(data_path)

class_counts = df["family"].value_counts()
class_pcts = df["family"].value_counts(normalize=True) * 100

dist_df = pd.DataFrame({
    "Count": class_counts,
    "Percentage (%)": class_pcts.round(2)
})
print("Class Distribution:")
display(dist_df)

# Plot class distribution
plt.figure(figsize=(7, 4))
ax = sns.barplot(x=class_counts.index, y=class_counts.values, palette=["#2b5c8f", "#e66101", "#5e3c99"])
plt.title("Ransomware Family Class Distribution", fontsize=12, fontweight="bold")
plt.xlabel("Malware Family", fontsize=10)
plt.ylabel("Number of Samples", fontsize=10)
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())} ({p.get_height()/len(df):.1%})",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha="center", va="bottom", fontsize=9, xytext=(0, 3), textcoords="offset points")
plt.tight_layout()
plt.show()


## 4. Duplicate Rows & Constant Feature Verification


In [ ]:
print(f"Duplicate rows count: {df.duplicated().sum()}")
features = [c for c in df.columns if c != "family"]
constant_cols = [c for c in features if df[c].nunique() <= 1]
print(f"Constant features count: {len(constant_cols)}")
assert df.duplicated().sum() == 0, "Error: Duplicate rows found!"
assert len(constant_cols) == 0, "Error: Constant features found!"


## 5. Feature Ranges and Summary Statistics


In [ ]:
print("Feature Summary Statistics (First 10 features):")
display(df[features[:10]].describe().T[["mean", "std", "min", "50%", "max"]])


## 6. Data Leakage & Identical Redundancy Audit
Execute full leakage investigation across the 14 suspicious features and 15 identical column pairs.


In [ ]:
# Run and display leakage analysis report
report_text = run_leakage_analysis()
print("Report excerpt:")
print(report_text[:2500])


## 7. Identical Column Verification
Confirm the 15 identical column pairs identified in the research report.


In [ ]:
identical_pairs = []
for i in range(len(features)):
    for j in range(i + 1, len(features)):
        c1, c2 = features[i], features[j]
        if (df[c1] == df[c2]).all():
            identical_pairs.append((c1, c2))

print(f"Confirmed {len(identical_pairs)} perfectly identical column pairs:")
for c1, c2 in identical_pairs:
    print(f"  * {c1:20s} == {c2:20s}")
